In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Tugas7").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

26/10/08 23:06:18 WARN Utils: Your hostname, nada resolves to a loopback address: 127.0.1.1; using 192.168.1.12 instead (on interface wlp1s0)
26/10/08 23:06:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/08 23:06:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# EXTRACT - MEMBACA
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang = spark.read.json("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json")
df_menu = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv", header=True, inferSchema=True)

print("Transaksi:", df_transaksi.count(), "| Cabang:", df_cabang.count(), "| Menu:", df_menu.count())

Transaksi: 4000 | Cabang: 3 | Menu: 15


In [3]:
# TRANSFORM - JOIN & KOLOM TURUNAN
df_gabungan = df_transaksi.join(df_cabang, on="cabang_id", how="inner") \
                           .join(df_menu, on="menu_id", how="inner")
df_gabungan = df_gabungan.withColumn("total_penjualan", col("qty") * col("harga"))

df_gabungan.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


In [4]:
# TRANSFORM - WINDOW FUNCTION TOP 2 MENU PER CABANG
ringkasan_menu_cabang = df_gabungan.groupBy("nama_cabang", "nama_menu", "kategori_menu").agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)
window_cabang = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())
top2_menu = ringkasan_menu_cabang.withColumn("peringkat", row_number().over(window_cabang)) \
                                   .filter(col("peringkat") <= 2)

top2_menu.orderBy("nama_cabang", "peringkat").show()

+--------------------+---------+--------------+---------------+---------+
|         nama_cabang|nama_menu| kategori_menu|total_penjualan|peringkat|
+--------------------+---------+--------------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|Makanan Ringan|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|Makanan Ringan|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|Makanan Ringan|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|Makanan Ringan|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|      Non-Kopi|        8160000|        2|
+--------------------+---------+--------------+---------------+---------+



In [6]:
# TRANSFORM - SPARK SQL TOTAL PENJUALAN PER KEPALA BARISTA
df_gabungan.createOrReplaceTempView("gabungan")

penjualan_barista = spark.sql('''
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM gabungan
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
''')

penjualan_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



In [7]:
# LOAD
!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/processed

df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"
)

print("Data berhasil disimpan ke HDFS, terpartisi per kategori_menu.")

df_cek = spark.read.parquet("hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan")
print("Jumlah baris hasil verifikasi:", df_cek.count())
!hdfs dfs -ls /user/mahasiswa/tugas7/processed/laporan

Data berhasil disimpan ke HDFS, terpartisi per kategori_menu.
Jumlah baris hasil verifikasi: 4000
Found 4 items
-rw-r--r--   3 nada supergroup          0 2026-10-08 23:07 /user/mahasiswa/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - nada supergroup          0 2026-10-08 23:07 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Kopi
drwxr-xr-x   - nada supergroup          0 2026-10-08 23:07 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Makanan Ringan
drwxr-xr-x   - nada supergroup          0 2026-10-08 23:07 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Non-Kopi


In [8]:
# RINGKASAN EKSEKUTIF
ringkasan_cabang = df_gabungan.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan").desc())

ringkasan_cabang.show()

+--------------------+----------+--------------+---------------+----------------+-------------------------+
|         nama_cabang|      kota|kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+--------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Se...|  Semarang|         Nadia|       90542000|            1393|         64997.8463747308|
|Kopi Nusantara Ma...|  Magelang|         Nadia|       87138000|            1302|        66926.26728110599|
|Kopi Nusantara Yo...|Yogyakarta|          Reza|       84645000|            1305|        64862.06896551724|
+--------------------+----------+--------------+---------------+----------------+-------------------------+



In [9]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
